<a href="https://colab.research.google.com/github/aashish003/BIS-LAB_1BF24CS003/blob/main/Grey_Wolf_optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import numpy as np

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# ------------------------------------------------
# STEP 1: Load Dataset
# ------------------------------------------------

data = load_breast_cancer()

X = data.data
y = data.target
feature_names = data.feature_names

N = X.shape[1]

print("Total Features:", N)

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# ------------------------------------------------
# STEP 2: Define Binary GWO Parameters
# ------------------------------------------------

wolves_count = 8
max_iterations = 15

# Initialize continuous wolf positions
wolves = np.random.uniform(-2, 2, (wolves_count, N))

# ------------------------------------------------
# STEP 3: Convert Positions into Binary Features
# ------------------------------------------------

def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -50, 50)))


def binary_position(position):
    probability = sigmoid(position)

    return (np.random.rand(N) < probability).astype(int)


# ------------------------------------------------
# STEP 4: Define Fitness Function
# ------------------------------------------------

def fitness(position):

    selected = np.where(position == 1)[0]

    # Avoid empty feature selection
    if len(selected) == 0:
        return 1.0

    X_selected = X_train[:, selected]

    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=2000)
    )

    # Evaluate using 3-fold cross validation
    scores = cross_val_score(
        model,
        X_selected,
        y_train,
        cv=3,
        scoring="accuracy"
    )

    error = 1 - scores.mean()

    feature_ratio = len(selected) / N

    # Weighted objective
    return 0.99 * error + 0.01 * feature_ratio


# ------------------------------------------------
# STEP 5: Grey Wolf Optimization
# ------------------------------------------------

for t in range(max_iterations):

    binary_wolves = np.array([
        binary_position(wolf) for wolf in wolves
    ])

    scores = np.array([
        fitness(wolf) for wolf in binary_wolves
    ])

    # Select Alpha, Beta and Delta
    order = np.argsort(scores)

    alpha = wolves[order[0]].copy()
    beta = wolves[order[1]].copy()
    delta = wolves[order[2]].copy()

    a = 2 - 2 * t / max_iterations

    new_wolves = np.zeros_like(wolves)

    # Update every wolf
    for i in range(wolves_count):

        candidate_positions = []

        for leader in [alpha, beta, delta]:

            r1 = np.random.rand(N)
            r2 = np.random.rand(N)

            A = 2 * a * r1 - a
            C = 2 * r2

            D = np.abs(C * leader - wolves[i])

            X_new = leader - A * D

            candidate_positions.append(X_new)

        new_wolves[i] = np.mean(
            candidate_positions, axis=0
        )

    wolves = np.clip(new_wolves, -6, 6)

    print(
        f"Iteration {t+1}: Best Fitness = "
        f"{scores[order[0]]:.4f}"
    )


# ------------------------------------------------
# STEP 6: Find Best Feature Subset
# ------------------------------------------------

final_binary = np.array([
    binary_position(wolf) for wolf in wolves
])

final_scores = np.array([
    fitness(wolf) for wolf in final_binary
])

best_index = np.argmin(final_scores)

best_features = final_binary[best_index]

selected_indices = np.where(best_features == 1)[0]

# ------------------------------------------------
# STEP 7: Compare Classification Performance
# ------------------------------------------------

# Model using all features
full_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000)
)

full_model.fit(X_train, y_train)

full_predictions = full_model.predict(X_test)

full_accuracy = accuracy_score(
    y_test, full_predictions
)

# Model using selected features
if len(selected_indices) == 0:
    raise RuntimeError("No features selected.")

selected_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000)
)

selected_model.fit(
    X_train[:, selected_indices], y_train
)

selected_predictions = selected_model.predict(
    X_test[:, selected_indices]
)

selected_accuracy = accuracy_score(
    y_test, selected_predictions
)

# ------------------------------------------------
# STEP 8: Display Results
# ------------------------------------------------

print("\n========== FINAL RESULTS ==========")

print("Original Features:", N)

print("Selected Features:", len(selected_indices))

print("\nSelected Feature Names:")

for index in selected_indices:
    print(feature_names[index])

print("\nAccuracy using all features:",
      round(full_accuracy * 100, 2), "%")

print("Accuracy using selected features:",
      round(selected_accuracy * 100, 2), "%")

print("Feature Reduction:",
      round((1 - len(selected_indices)/N) * 100, 2), "%")

print("Final Fitness:",
      round(final_scores[best_index], 4))


Total Features: 30
Iteration 1: Best Fitness = 0.0264
Iteration 2: Best Fitness = 0.0249
Iteration 3: Best Fitness = 0.0274
Iteration 4: Best Fitness = 0.0176
Iteration 5: Best Fitness = 0.0164
Iteration 6: Best Fitness = 0.0230
Iteration 7: Best Fitness = 0.0295
Iteration 8: Best Fitness = 0.0320
Iteration 9: Best Fitness = 0.0218
Iteration 10: Best Fitness = 0.0217
Iteration 11: Best Fitness = 0.0242
Iteration 12: Best Fitness = 0.0224
Iteration 13: Best Fitness = 0.0237
Iteration 14: Best Fitness = 0.0217
Iteration 15: Best Fitness = 0.0242

========== FINAL RESULTS ==========
Original Features: 30
Selected Features: 12

Selected Feature Names:
mean texture
mean area
mean concavity
mean symmetry
mean fractal dimension
texture error
perimeter error
worst radius
worst texture
worst perimeter
worst smoothness
worst compactness

Accuracy using all features: 98.83 %
Accuracy using selected features: 97.66 %
Feature Reduction: 60.0 %
Final Fitness: 0.0189
